In [34]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from pickle import load
from pathlib import Path

import re
from tqdm import tqdm

import matplotlib as mpl
import seaborn as sns

mpl.rcParams.update({
    # Use LaTeX
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],

    # Font sizes
    "font.size": 16,
    "axes.labelsize": 20,
    "axes.titlesize": 16,
    "legend.fontsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,

    # Lines
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 1,

    # Ticks
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,

    # Save figures
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})

In [35]:
ROOT = Path.cwd()
if not (ROOT / 'data' / 'single_dot_error.pickle').exists() and (ROOT / 'charge_stability' / 'data' / 'single_dot_error.pickle').exists():
    ROOT = ROOT / 'charge_stability'

DATA_PATH = ROOT / 'data'
FIGURE_PATH = ROOT / 'Figures'
FIGURE_PATH.mkdir(exist_ok=True)

RESULTS_FILE = DATA_PATH / 'single_dot_error.pickle'
with RESULTS_FILE.open('rb') as f:
    single_dot_error_results = load(f)

single_dot_error_results['metadata']

{'description': 'Closed-shell charge stability sweep for a double quantum dot.',
 'length_scale_nm': 50.0,
 'effective_units_reference_energy': 2.5702165936004097e-23,
 'coulomb_energy_effective_units': 15.34392984483895,
 'eV_error_tolerance': 1e-07,
 'e_conv_tolerance_effective_units': 0.0006233624971487868,
 'log_file': 'data/single_dot_error.log'}

# Time Analysis of Single Dot Results

In [44]:
records = []

for run in single_dot_error_results['runs']:
    n_electrons = run['n_electrons']
    run_voltage = run.get('V')

    for point in run['points']:
        row = {key: value for key, value in point.items() if key != 'profiler'}
        row['N'] = n_electrons
        row['run_v'] = run_voltage if run_voltage is not None else point.get('v')
        row['profiler'] = point['profiler']
        records.append(row)

results_df = (
    pd.DataFrame(records)
    .sort_values(['N', 'n_orbitals'])
    .reset_index(drop=True)
)

# results_df[['N', 'run_v', 'n_orbitals', 'energy', 'n_iterations', 'converged', 'log_time']]

In [52]:
def profile_stage_info(stage_name):
    iteration = None

    match = re.search(r'for iteration (\d+)$', stage_name)
    if match is None:
        iterative_prefixes = (
            'Integrals ',
            'FCI ',
            'Orbital Refinement ',
            'Two-body integrals transformation ',
            'RDM2 transformation ',
        )
        if stage_name.startswith(iterative_prefixes):
            match = re.search(r'(\d+)$', stage_name)

    if match is not None:
        iteration = int(match.group(1))

    if stage_name == 'MAD world initialization':
        stage = 'MAD world initialization'
    elif stage_name in {'MRA potential creation', 'MRA function factory initialization'}:
        stage = 'MRA potential creation'
    elif stage_name == 'Single-particle orbitals calculation':
        stage = 'Single-particle orbitals'
    elif stage_name == 'Integral computation':
        stage = 'Initial integral setup'
    elif stage_name == 'Two-body integrals computation':
        stage = 'Two-body integral setup'
    elif stage_name.startswith('Integrals computation for iteration') or stage_name.startswith('Integrals '):
        stage = 'SCF integrals'
    elif stage_name.startswith('FCI calculation for iteration') or stage_name.startswith('FCI '):
        stage = 'FCI'
    elif stage_name.startswith('Two-body integrals transformation'):
        stage = 'Two-body transform'
    elif stage_name.startswith('RDM2 transformation'):
        stage = 'RDM2 transform'
    elif stage_name.startswith('Orbital Refinement for iteration') or stage_name.startswith('Orbital Refinement '):
        stage = 'Orbital refinement'
    else:
        stage = stage_name

    phase = 'SCF iteration' if iteration is not None else 'Setup'
    return stage, iteration, phase


profile_records = []

for _, row in results_df.iterrows():
    for stage_name, metrics in row['profiler'].items():
        stage, iteration, phase = profile_stage_info(stage_name)
        profile_records.append({
            'N': row['N'],
            'v': row['run_v'],
            'n_orbitals': row['n_orbitals'],
            'iteration': iteration,
            'phase': phase,
            'stage': stage,
            'raw_stage': stage_name,
            'time_s': metrics['time_s'],
            'memory_change_MB': metrics['memory_change_MB'],
        })

profile_df = pd.DataFrame(profile_records)

time_by_stage = (
    profile_df
    .groupby(['N', 'v', 'n_orbitals', 'phase', 'stage'], as_index=False, dropna=False)
    .agg(time_s=('time_s', 'sum'), memory_change_MB=('memory_change_MB', 'sum'))
)

stage_order = [
    'MAD world initialization',
    'MRA potential creation',
    'Single-particle orbitals',
    'Initial integral setup',
    'Two-body integral setup',
    'SCF integrals',
    'Two-body transform',
    'FCI',
    'RDM2 transform',
    'Orbital refinement',
]

stage_order = [stage for stage in stage_order if stage in time_by_stage['stage'].unique()]

## Per-Iteration Timing Statistics

In [53]:
iteration_time_by_stage = (
    profile_df[profile_df['iteration'].notna()]
    .assign(iteration=lambda df: df['iteration'].astype(int))
    .groupby(['N', 'v', 'n_orbitals', 'iteration', 'stage'], as_index=False)
    .agg(time_s=('time_s', 'sum'), memory_change_MB=('memory_change_MB', 'sum'))
)

iteration_time_table = (
    iteration_time_by_stage
    .pivot_table(
        index=['N', 'v', 'n_orbitals', 'iteration'],
        columns='stage',
        values='time_s',
        aggfunc='sum',
        fill_value=0.0,
    )
    .reset_index()
)

stage_columns = [col for col in iteration_time_table.columns if col not in {'N', 'v', 'n_orbitals', 'iteration'}]
iteration_time_table['total_iteration_time_s'] = iteration_time_table[stage_columns].sum(axis=1)
iteration_time_table = iteration_time_table.sort_values(['N', 'n_orbitals', 'iteration']).reset_index(drop=True)

iteration_time_table.head(5)

stage,N,v,n_orbitals,iteration,FCI,Orbital refinement,RDM2 transform,SCF integrals,Two-body transform,total_iteration_time_s
0,2,0.4,2,0,0.087693,0.712353,0.0,0.070538,0.0,0.870583
1,2,0.4,2,1,0.075146,0.797390,0.0,0.070217,0.0,0.942753
2,2,0.4,2,2,0.087347,0.590933,0.0,0.068962,0.0,0.747242
3,2,0.4,2,3,0.078529,0.256824,0.0,0.069239,0.0,0.404591
4,2,0.4,2,4,0.071175,0.298488,0.0,0.069690,0.0,0.439353


In [51]:
def summarize_iteration_times(df, group_cols):
    stats = (
        df
        .groupby(group_cols, as_index=False)
        .agg(
            n_iteration_samples=('time_s', 'count'),
            total_time_s=('time_s', 'sum'),
            mean_time_s=('time_s', 'mean'),
            median_time_s=('time_s', 'median'),
            std_time_s=('time_s', 'std'),
            min_time_s=('time_s', 'min'),
            q25_time_s=('time_s', lambda values: values.quantile(0.25)),
            q75_time_s=('time_s', lambda values: values.quantile(0.75)),
            max_time_s=('time_s', 'max'),
        )
    )
    stats['std_time_s'] = stats['std_time_s'].fillna(0.0)
    stats['iqr_time_s'] = stats['q75_time_s'] - stats['q25_time_s']
    return stats


iteration_stage_stats_by_orbital = summarize_iteration_times(
    iteration_time_by_stage,
    ['N', 'n_orbitals', 'stage'],
)


iteration_stage_stats_by_orbital

,N,n_orbitals,stage,n_iteration_samples,total_time_s,mean_time_s,median_time_s,std_time_s,min_time_s,q25_time_s,q75_time_s,max_time_s,iqr_time_s
0,2,2,FCI,5,0.399891,0.079978,0.078529,0.007361,0.071175,0.075146,0.087347,0.087693,0.012201
1,2,2,Orbital refinement,5,2.655987,0.531197,0.590933,0.243248,0.256824,0.298488,0.712353,0.797390,0.413865
2,2,2,SCF integrals,5,0.348645,0.069729,0.069690,0.000656,0.068962,0.069239,0.070217,0.070538,0.000978
3,2,4,FCI,13,0.896906,0.068993,0.065384,0.020739,0.041994,0.056409,0.071028,0.107282,0.014619
4,2,4,Orbital refinement,13,47.775661,3.675051,3.588539,0.366185,3.364620,3.569535,3.599237,4.861462,0.029701
...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,3,19,FCI,6,24.089938,4.014990,4.050746,0.165400,3.784020,3.893297,4.154971,4.173900,0.261674
65,3,19,Orbital refinement,6,14191.345551,2365.224258,2365.928699,46.837958,2305.092743,2334.862104,2386.110232,2437.051427,51.248128
66,3,19,RDM2 transform,6,0.001618,0.000270,0.000268,0.000009,0.000261,0.000263,0.000275,0.000282,0.000012
67,3,19,SCF integrals,6,247.886503,41.314417,42.307816,8.306937,25.939749,40.738424,46.394862,49.358618,5.656439
